# Creating Raster Information Product using Raster Analytics

## Raster Analytics

ArcGIS Enterprise at 10.5 provides you with the ability to perform large raster data analytics using distributed computing using raster analytics tools. This analytics capability is provided in the `arcgis.raster.analytics` module and includes functionality to summarize data, analyze patterns, images, terrain and manage data. This sample show the capabilities of imagery layers and raster analytics.

## Imagery layers

In [1]:
import arcgis
from arcgis.gis import GIS
from IPython.display import display

gis = GIS()

Here we're searchcing for multispectral landsat imagery layer:

In [2]:
items = gis.content.search("Landsat 8 Views", max_items=2)

In [3]:
for item in items:
    display(item)

<Item title:"Imagery" type:Layer owner:esri>

<Item title:"Landsat 8 Views" type:Image Service owner:esri>

In [4]:
landast_item = items[1]

In [5]:
imglyr = landast_item.layers[0]

Let us create a map widget and load this layer.

In [6]:
marthasbasin = arcgis.geocoding.geocode("Marthas Basin, Montana")[0]

In [13]:
map1 = gis.map(marthasbasin, zoomlevel=13)

In [14]:
map1

In [15]:
map1.add_layer(imglyr)

Here we are interactively cycling through these raster functions published with the layer and apply them to the map. This is using on-the-fly image processing at display resolution to cycle through the various raster functions, and showing how the visualization of the layer changes.

In [16]:
import time

for fn in imglyr.properties['rasterFunctionInfos'][:6]:
    print(fn['name'])
    map1.remove_layers()
    map1.add_layer(imglyr, {"imageServiceParameters" :{ "renderingRule": { "rasterFunction": fn['name']}}})
    time.sleep(2)
    

Agriculture with DRA
Bathymetric with DRA
Color Infrared with DRA
Natural Color with DRA
Short-wave Infrared with DRA
Geology with DRA


## Raster functions

Developers can create their own **raster functions**, by chaining different raster functions. For instance, the code below is doing an Extract Band and extracting out the [4,5,3] band combination, and applying a Stretch to get the land-water boundary visualization that makes it easy to see where land is and where water is. Its worth noting that the raster function is applied at display resolution and only for the visible extent using on the fly image processing.

In [7]:
from arcgis.raster.functions import stretch, extract_band

In [8]:
land_water_viz = stretch(extract_band(imglyr, band_ids=[4, 5, 3]), 
                         min_percent=1, max_percent=1, gamma=[1,1,1], dra=True)        

Let us apply this raster function to the image layer to visualize the results.

In [29]:
map2 = gis.map(marthasbasin, zoomlevel=13)
map2

In [30]:
map2.add_layer(land_water_viz)

# Creating a Raster Information Product using Landsat 8 imagery

This part of the notebook shows how **Raster Analytics** (in ArcGIS Enterprise 10.5) can be used to generate a raster information product, by applying the same raster function across the extent of an image service on the portal. The raster function is applied at source resolution and creates an Information Product, that can be used for further analysis and visualization.

In [9]:
portal = GIS("https://dev003246.esri.com/portal", "admin","esri.agp")

In [10]:
montana_landsat = portal.content.search("ImgSrv_Landast_Montana2015")[0]
montana_lyr = montana_landsat.layers[0]

We can use the `arcgis.raster.analytics.generate_raster()` tool to apply the raster function across the entire extext of the input image layer at source resolution, and presist the result in another output image layer. This creates a raster product similar that can be used for further analysis and visualization.

In the code below, we use a raster function that extracts the [7, 5, 2] band combination. This improves visibility of fire and burn scars by pushing further into the SWIR range of the electromagnetic spectrum, as there is less susceptibility to smoke and haze generated by a burning fire.

In [11]:
fire_viz = extract_band(montana_lyr, band_ids=[7, 5, 2])  

In [13]:
import json
json.dumps(fire_viz._fnra)

'{"rasterFunction": "ExtractBand", "variableName": "Raster", "rasterFunctionArguments": {"BandIDs": [7, 5, 2], "Raster": "https://dev003248.esri.com/rax/rest/services/ImgSrv_Landast_Montana2015/ImageServer"}}'

In [14]:
montana_fires = fire_viz.save()

Submitted.
Executing...
Executing (GenerateRaster): GenerateRaster {"rasterFunction":"ExtractBand","variableName":"Raster","rasterFunctionArguments":{"BandIDs":[7,5,2],"Raster":"https://dev003248.esri.com/rax/rest/services/ImgSrv_Landast_Montana2015/ImageServer"}} {"serviceProperties":{"serviceUrl":"http://dev003248.esri.com/rax/rest/services/Hosted/GeneratedRasterProduct_O2VORN/ImageServer","name":"GeneratedRasterProduct_O2VORN"},"itemProperties":{"itemId":"4d9028e02d424fe0a49e2a0d688b0f33"}} # # #
Start Time: Fri Apr 21 10:44:41 2017
Running script GenerateRaster...
Image service GeneratedRasterProduct_O2VORN already existed.
GetPrivateUrl returns: https://dev003248.esri.com:6443/arcgis/rest/services/Hosted/GeneratedRasterProduct_O2VORN/ImageServer
The service got from item ID is: https://dev003248.esri.com:6443/arcgis/rest/services/Hosted/GeneratedRasterProduct_O2VORN/ImageServer
Output item id is: 4d9028e02d424fe0a49e2a0d688b0f33
Output image service url is: https://dev003248.esri.

In [15]:
montana_fires

<Item title:"GeneratedRasterProduct_O2VORN" type:Image Service owner:admin>

In [16]:
location = arcgis.geocoding.geocode("Marthas Basin, Montana")[0]

base_map = portal.map(location, 12)

natural_color_map = portal.map(location, 12)
natural_color_map.add_layer(montana_landsat)

false_color_map = portal.map(location, 12)
false_color_map.add_layer(montana_fires)

In [17]:
import ipywidgets as widgets

tab = widgets.Tab([base_map, natural_color_map, false_color_map])
tab.set_title(0, 'Basemap')
tab.set_title(1, 'Natural Color')
tab.set_title(2, 'False Color')
tab

We can compare the natural color and false color images uaing a tabbed widget. 

In the false color image the red and brownish pixels correspond to burn scars of the fire:

Thus using the same raster function, we were able to both visualize on the fly (in the case of Pallikaranai marsh example) and also derive a persisted image layer (in the case of Montana example) with the power of raster analytics.